# 05 — Text classification (the demanding case)

*Module 02 · Naive Bayes — notebook 5 of 5*

This is the chapter's proving ground — and Naive Bayes on its home turf. In chapter 01, k-NN drowned in
high dimensions: with thousands of features every point looked equally far from every other (the curse
of dimensionality). Naive Bayes never measures a distance — it counts words — so high-dimensional,
sparse **text** is exactly where it shines. We run the full honest workflow on a real corpus of
newsgroup posts: turn text into numbers, fit the classifier, evaluate it honestly under class imbalance,
and finally put its confidence on trial.

**Prerequisites:** notebooks 01–04 (the whole Naive Bayes story); module 00 notebooks 07–08 (confusion
matrix, precision/recall/F1, the precision-recall curve); chapter 01 (the curse of dimensionality).

**What you will be able to do**

- Turn raw text into a numeric **bag-of-words** matrix, by hand and with `CountVectorizer`.
- Fit `MultinomialNB` on thousands of sparse features and read its errors.
- Evaluate **honestly under imbalance** — precision, recall, F1, and the PR curve, not accuracy alone.
- Judge whether to trust Naive Bayes' **probabilities**, not only its ranking.

In [ ]:
import logging
import time

import matplotlib.pyplot as plt
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    brier_score_loss,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
)
from sklearn.naive_bayes import MultinomialNB

from ml_course import datasets, viz
from ml_course.colors import CLASS_CYCLE, CMAP_COUNT, COLORS

np.random.seed(0)
viz.use_course_style()
logging.basicConfig(level=logging.INFO)  # show the dataset's fetch/cache logging (never silenced)

CATS = ["comp.graphics", "rec.sport.baseball", "sci.med", "talk.religion.misc"]
train = datasets.load_newsgroups(CATS, subset="train")
test = datasets.load_newsgroups(CATS, subset="test")
print(f"\ntrain documents: {len(train)}   test documents: {len(test)}")
print(train["category"].value_counts())
print("\nexample sci.med document (first 300 chars):")
print(train[train["category"] == "sci.med"]["text"].iloc[0][:300])

## Why text, and why Naive Bayes

A newsgroup post draws on thousands of possible words, and any one post uses only a handful of them. In
chapter 01 that kind of high-dimensional, sparse space was fatal for k-NN: distances stopped
discriminating. Naive Bayes has no such weakness, because it never computes a distance — it asks, for
each class, *how probable are these word counts?* and multiplies. So we leave the two-feature penguins
behind for a genuinely hard problem: sorting posts into topics.

## From text to numbers, by hand

A classifier needs numbers, not strings. The standard representation for Naive Bayes is the **bag of
words**: fix a **vocabulary** (the set of words that appear), then describe each document by a vector of
**word counts** — how many times each vocabulary word occurs. Word order is thrown away (hence "bag").
Let us build one by hand on four tiny sentences before turning a library loose on thousands.

In [ ]:
docs = ["the cat sat", "the dog ran", "cats and dogs", "the cat ran fast"]
vocabulary = sorted({word for d in docs for word in d.split()})
counts = np.array([[d.split().count(w) for w in vocabulary] for d in docs])
print("documents:", docs)
print("vocabulary:", vocabulary)

fig, ax = plt.subplots(figsize=(8, 3.0))
im = ax.imshow(counts, cmap=CMAP_COUNT, aspect="auto")
for i in range(counts.shape[0]):
    for j in range(counts.shape[1]):
        ax.text(j, i, counts[i, j], ha="center", va="center", color=COLORS["text"])
ax.set_xticks(range(len(vocabulary)), vocabulary, rotation=45, ha="right")
ax.set_yticks(range(len(docs)), [f"doc {i + 1}" for i in range(len(docs))])
ax.set_title("A toy document-term matrix (word counts)")
ax.grid(False)
plt.show()

**Read the figure.** This is a **document-term matrix**: one row per document, one column per
vocabulary word, each cell the count of that word in that document. "doc 1" (*the cat sat*) has a 1 under
*cat*, *sat*, and *the*, and 0 elsewhere. It is exactly the counting from notebook 1 — now one count per
word, across the whole vocabulary. Most cells are zero, because each short document uses few words.

## At scale: `CountVectorizer`

`CountVectorizer` does precisely what we did by hand, for thousands of documents at once: it scans the
training texts, builds the vocabulary, and returns the document-term matrix. Two settings keep it
sensible: `stop_words="english"` drops uninformative words ("the", "is"), and `min_df=2` ignores words
seen in only a single document. Crucially we **fit it on the training set only** — letting it see the
test documents would leak information (the rule from module 00) — then *transform* the test set with that
fixed vocabulary.

In [ ]:
vectorizer = CountVectorizer(stop_words="english", min_df=2)
X_train = vectorizer.fit_transform(train["text"])   # fit on TRAIN only
X_test = vectorizer.transform(test["text"])

density = X_train.nnz / (X_train.shape[0] * X_train.shape[1])
print(f"vocabulary size (fit on train): {len(vectorizer.vocabulary_)}")
print(f"document-term matrix: {X_train.shape[0]} docs x {X_train.shape[1]} words")
print(f"non-zero entries: {X_train.nnz}  ({density:.4f} of the matrix is filled)")

**Read the output.** The training vocabulary is about 12 000 words, so each document is a vector in
~12 000 dimensions — and only about **0.4 %** of the entries are non-zero. That is the **sparse,
high-dimensional** world where k-NN's distances collapsed. Naive Bayes does not mind: it only ever sums
the log-probabilities of the words a document actually contains — with the multinomial count model we fit
next.

## MultinomialNB: the count likelihood at scale

For word counts the right likelihood is the **multinomial** (notebook 4 named it). It models, per class,
how probable each word is — fit by counting words across that class's documents, with `alpha` smoothing
so an unseen word never zeroes the product (notebook 1's zero-frequency trap, at scale). The prediction
is notebook 1's **argmax** rule, now over four classes and summed in log-space (notebook 3).

In [ ]:
t0 = time.perf_counter()
nb = MultinomialNB().fit(X_train, train["category"])
fit_ms = (time.perf_counter() - t0) * 1e3
pred = nb.predict(X_test)

print(f"MultinomialNB trained in {fit_ms:.1f} ms")
print(f"test accuracy {accuracy_score(test['category'], pred):.3f}   "
      f"macro-F1 {f1_score(test['category'], pred, average='macro'):.3f}")

short = ["graphics", "baseball", "med", "religion"]   # short labels, in CATS order
cm = confusion_matrix(test["category"], pred, labels=CATS)
viz.plot_confusion_matrix(cm, short)
plt.show()

**Read the figure.** Naive Bayes reaches about **0.89 accuracy**, and it trained in
**milliseconds** on twelve thousand features — fast where k-NN would crawl. The confusion matrix shows
where it struggles: `talk.religion.misc` is the hardest class, bleeding into the others (religious posts
share vocabulary with many topics), while baseball and graphics are cleanly separated. Strong and instant
on data where distances fail — this is why Naive Bayes is still a serious text baseline.

## Honest evaluation under imbalance

Accuracy can flatter a classifier when one class dominates. To see this honestly, reframe the task as a
single rare topic against everything else: **`sci.med` vs the rest**. Now most documents are "rest", so a
lazy model could score high by always guessing "rest". We bring in the metrics from module 00 that
respect the minority class.

In [ ]:
y_train_med = (train["category"] == "sci.med").astype(int)
y_test_med = (test["category"] == "sci.med").astype(int)

nb_med = MultinomialNB().fit(X_train, y_train_med)
pred_med = nb_med.predict(X_test)
proba_med = nb_med.predict_proba(X_test)[:, 1]

baseline = 1 - y_test_med.mean()   # accuracy of always guessing the majority ("rest")
print(f"test: {int(y_test_med.sum())} sci.med vs {int((1 - y_test_med).sum())} rest")
print(f"accuracy {accuracy_score(y_test_med, pred_med):.3f}   (majority baseline {baseline:.3f})\n")
print(classification_report(y_test_med, pred_med, target_names=["rest", "sci.med"]))

precision, recall, _ = precision_recall_curve(y_test_med, proba_med)
ap = average_precision_score(y_test_med, proba_med)
fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(recall, precision, color=COLORS["model"], linewidth=2, label=f"sci.med (AP = {ap:.3f})")
ax.axhline(y_test_med.mean(), color=COLORS["muted"], linestyle="--", linewidth=1,
           label=f"no-skill = positive rate ({y_test_med.mean():.2f})")
ax.set_xlabel("recall")
ax.set_ylabel("precision")
ax.set_ylim(0, 1.02)
ax.set_title("Precision-recall: sci.med vs the rest")
ax.legend(loc="lower left")
plt.show()

**Read the figure.** The accuracy (about 0.93) looks excellent — until you see the **majority
baseline** beside it (about 0.72): always guessing "rest" already scores 0.72, so accuracy alone is
misleading here. The honest picture is the per-class **precision and recall** and the **precision-recall
curve**: Naive Bayes finds most `sci.med` posts (recall) while keeping its `sci.med` calls mostly correct
(precision), with an average precision far above the no-skill line. This is the discipline from module 00,
on a real imbalanced problem.

## Can we trust the probabilities? Calibration

Notebook 4 warned that Naive Bayes can be **over-confident**. A model is **calibrated** if its
probabilities mean what they say: of all the documents it calls "90 % `sci.med`", about 90 % really
should be `sci.med`. Let us put that confidence on trial — against logistic regression as a reference —
with two views: the spread of the predicted probabilities, then a **reliability diagram**.

In [ ]:
lr = LogisticRegression(max_iter=2000).fit(X_train, y_train_med)
proba_lr = lr.predict_proba(X_test)[:, 1]

fig, axes = plt.subplots(1, 2, figsize=(12, 4.3))
for ax, name, pp in [(axes[0], "MultinomialNB", proba_med), (axes[1], "LogReg", proba_lr)]:
    ax.hist(pp, bins=25, color=CLASS_CYCLE[0], edgecolor=COLORS["text"], linewidth=0.4)
    extreme = int(((pp > 0.99) | (pp < 0.01)).sum())
    ax.set_title(f"{name}\n{extreme} of {len(pp)} predictions past 0.99 / 0.01")
    ax.set_xlabel("predicted P(sci.med)")
    ax.set_ylabel("count")
fig.tight_layout()
plt.show()

**Read the figure.** Naive Bayes crushes most of its probabilities to **0 or 1** — far more than
logistic regression does (1205 of 1433 vs 534). That is over-confidence **in shape**: because it treats
co-occurring words as independent, it double-counts the evidence and stakes its certainty at the
extremes. Logistic regression keeps more of a spread. But "more spread" is not the same as "better
calibrated" — whether those confident calls are *earned* is what the reliability diagram tests next.

In [ ]:
fig = viz.plot_calibration_curve(y_test_med, proba_med, label="MultinomialNB")
viz.plot_calibration_curve(
    y_test_med, proba_lr, label="LogReg", ax=fig.axes[0], color=COLORS["highlight"]
)
fig.axes[0].set_title("Reliability diagram (sci.med vs the rest)")
plt.show()

print(f"Brier score   MultinomialNB {brier_score_loss(y_test_med, proba_med):.4f}   "
      f"LogReg {brier_score_loss(y_test_med, proba_lr):.4f}")

**Read the figure.** The reliability diagram checks whether those confident calls are *earned*: it plots,
for each band of predicted probability, the fraction that were really `sci.med`. Here — on a fairly easy
split — both models stay within a reasonable band of the diagonal, and Naive Bayes is actually a touch
*closer*, which is why its Brier score edges out logistic regression's. So the honest lesson is **not**
"its score is worse." The warning is the **pile-up at 0/1** from the previous figure: Naive Bayes commits
almost everything to certainty, and that bet pays off here only because the task is easy. Trust its
**ranking** of documents; recalibrate if you genuinely need the probability — because on a harder, more
confusable task the same over-confidence turns into confidently *wrong* answers (see *Your turn*).

## A wrong assumption that still works

Step back at the absurdity: in text, words co-occur constantly — "hiv" travels with "patients", "season"
with "team" — so the assumption that words are independent given the topic is *wildly* false. The feature
space is enormous and sparse, exactly where k-NN died of the curse. And yet Naive Bayes is fast, strong,
and a standard baseline. This is the lesson from notebook 2, now at full scale: a wrong assumption can
still make a right decision (Domingos & Pazzani, 1997), because the prediction needs only the right
*ranking* of the classes, not exact probabilities.

**When to use Naive Bayes:** high-dimensional, sparse data (text above all); a fast, strong baseline;
small training sets. **When not:** when you need trustworthy probabilities, or when the dependence
between features genuinely carries the signal.

## A bridge to the next chapter: generative vs discriminative

Naive Bayes is a **generative** model: it learns how each class *generates* its data —
P(words ∣ topic) — then flips that around with Bayes' rule to get P(topic ∣ words). The next method,
**logistic regression** (module 03), takes the other road: it models P(topic ∣ words) **directly**,
without ever describing how the words are generated. That contrast — generative vs discriminative — is one
of the organizing ideas of machine learning (Ng & Jordan, 2001), and you now have a worked example of the
generative side to compare against.

## Your turn

1. **What did it learn?** From the fitted model's `feature_log_prob_`, print the words most distinctive of
   `sci.med`. Do they look medical? (This is how you sanity-check a text classifier.)
2. **When over-confidence bites.** Load the genuinely confusable pair `alt.atheism` vs
   `talk.religion.misc`, fit Naive Bayes and logistic regression, and compare their Brier scores. Here the
   over-confidence should finally **cost** Naive Bayes (its Brier turns worse than logistic regression's).
3. **(Harder)** Swap `CountVectorizer` for `TfidfVectorizer`, or try `ComplementNB`. Does the accuracy
   move? Does the pile-up of probabilities at 0 and 1 soften?

## What you built

You ran Naive Bayes end to end on real text. You turned documents into a **bag-of-words** matrix (by hand,
then with `CountVectorizer`, fit on train only), fit **`MultinomialNB`** on twelve thousand sparse
features in milliseconds, and read its errors. You evaluated it **honestly under imbalance** — precision,
recall, F1, and the PR curve against the majority baseline, never accuracy alone. And you put its
confidence on trial: Naive Bayes is a strong **ranker** but an over-confident, poorly **calibrated**
source of probabilities — trust the order, not the number.

**Vocabulary**

- **bag of words / document-term matrix** — text as per-document word counts; word order discarded.
- **sparse, high-dimensional** — thousands of features, almost all zero per document; where distances fail
  and counting thrives.
- **calibration / reliability diagram / Brier score** — whether predicted probabilities can be read as
  probabilities; the diagram and the Brier score measure it.
- **generative vs discriminative** — modelling P(features ∣ class) (Naive Bayes) vs P(class ∣ features)
  (logistic regression, next).

## What you built across the chapter

That completes Naive Bayes. Over five notebooks you built it from nothing:

- **01** — Bayes' rule from counts: prior, likelihood, posterior, argmax.
- **02** — the "naive" assumption (conditional independence): what it buys, where it breaks, why it still
  works.
- **03** — the Gaussian likelihood for continuous features, computed safely in log-space.
- **04** — the real estimators and their dials: `var_smoothing`, `alpha`, `priors`, and honest tuning.
- **05** — text, the demanding case: bag-of-words, `MultinomialNB`, honest evaluation, and calibration.

You can now build a Naive Bayes classifier by hand, drive the library version, evaluate it honestly, and
say clearly when to reach for it — and when not to.

## Going further (optional)

- **Better text features.** `TfidfVectorizer` down-weights common words; `ComplementNB` (Rennie et al.,
  2003) corrects Naive Bayes' bias on imbalanced text. Both often beat the plain count model.
- **Trustworthy probabilities.** `CalibratedClassifierCV` wraps a classifier and recalibrates its scores —
  the practical fix for the over-confidence we saw.
- **Beyond single words.** `ngram_range` lets the vocabulary include short phrases ("not good"), recovering
  some of the word-order information the bag of words throws away.

## References

- C. D. Manning, P. Raghavan, H. Schütze (2008). *Introduction to Information Retrieval*, ch. 13 (text
  classification & Naive Bayes). DOI:
  [10.1017/CBO9780511809071](https://doi.org/10.1017/CBO9780511809071).
- J. D. M. Rennie, L. Shih, J. Teevan, D. R. Karger (2003). *Tackling the poor assumptions of naive Bayes
  text classifiers.* ICML-2003, 616–623.
- A. McCallum, K. Nigam (1998). *A comparison of event models for naive Bayes text classification.*
  AAAI-98 Workshop on Learning for Text Categorization.
- A. Y. Ng, M. I. Jordan (2001). *On discriminative vs. generative classifiers.* NeurIPS 14:841–848.
- A. Niculescu-Mizil, R. Caruana (2005). *Predicting good probabilities with supervised learning.*
  ICML-2005. DOI: [10.1145/1102351.1102430](https://doi.org/10.1145/1102351.1102430).
- ISLR (James et al., 2021) §4.4.4. DOI:
  [10.1007/978-1-0716-1418-1](https://doi.org/10.1007/978-1-0716-1418-1).

---

*Previous: 04 — The estimators & their parameters* · *Next: Module 03 — Logistic Regression*